# pick_place: motion primitives and chess-move sequences

Library notebook for `gambit_arm`. Builds every arm motion out of a few primitives
(move_to_hover, descend, grasp, lift, place/release, park), composes them into `pick`,
`place`, `move_piece` and `capture_to_graveyard`, and expands a chess move (including
castling, en passant and promotion) into those. A trajectory is a plain list of waypoints;
`plan_joint_trajectory` turns it into joint targets with IK and checks each against safety.
Execution backends (dry run, real servos, Gazebo controller, LeRobot) are TODO.

Milestone: *Hardware and execution* (pick-and-place success 55-60 % first attempt, 95 % with
verify-and-retry, see run_pick_place_trials.ipynb).

**TODO**
- [ ] Cartesian waypoint primitives from board.yaml (hover height, grasp height fraction)
- [ ] pick / place / move_piece / capture_to_graveyard, chess move -> sequence
- [ ] IK + safety check per waypoint (`plan_joint_trajectory`), dry-run backend
- [ ] joint-space interpolation (`densify`) with every step safety-checked
- [ ] straight-line descend/lift in Cartesian space, timing from safety.yaml velocity limits
- [ ] backends: `ServoBackend` (servo_bus.ipynb), `GazeboBackend` (FollowJointTrajectory on
      /arm_controller and /gripper_controller, see docs/gazebo.md), `LeRobotBackend`
- [ ] promotion with a spare queen from the graveyard; under-promotion
- [ ] approach tilt chosen per waypoint from a candidate list (`solve_waypoint`)
- [ ] rank 1 (c1-f1) hover is out of reach with the current limits, see the reachability note
- [ ] finger orientation (wrist_roll) so the jaws close across the piece, not along a file
- [ ] tune descend speed and grasp pause on the real arm
- [ ] write the self-checks at the bottom as asserts

In [ ]:
import math
from dataclasses import dataclass, field
import numpy as np

%run ./safety.ipynb

## Waypoints

A waypoint is either Cartesian (`xyz` of the gripper tip, plus a tilt from vertical) or a
joint-space pose (`q`, used for park). `gripper` is the gripper joint target to hold while
moving to it (board.yaml `gripper_open` / `gripper_closed`). `slow` marks segments near the
pieces (descend, lift) that must use the reduced Cartesian speed.

In [ ]:
ARM = BOARD["arm"]

@dataclass
class Waypoint:
    name: str
    xyz: np.ndarray = None
    q: np.ndarray = None
    gripper: float = ARM["gripper_open"]
    tilt_deg: float = 0.0
    slow: bool = False

    def __post_init__(self):
        # Design: exactly one of xyz / q must be given (Cartesian or joint-space waypoint);
        # fail at construction time, not deep inside the planner.
        # TODO: implement the check
        raise NotImplementedError

## Primitives

Each returns a list of waypoints. `target` is a square name or a world point (graveyard
slot). The gripper state is carried along: e.g. `lift` after `grasp` keeps it closed.

In [ ]:
def move_to_hover(target, gripper=ARM["gripper_open"]) -> list:
    """[Waypoint] at hover_point(target), gripper held at `gripper`. Not slow (free transit)."""
    # Design: one waypoint; name it "hover <square>" (tests and logs read the names).
    # TODO: implement
    raise NotImplementedError

def descend(target, kind: str, gripper=ARM["gripper_open"]) -> list:
    """[Waypoint] down to grasp_point(target, kind), slow."""
    # TODO: implement
    raise NotImplementedError

def grasp(target, kind: str) -> list:
    """Close the gripper in place at the grasp point."""
    # Design: same xyz as descend, gripper_closed, slow. A separate waypoint (not just a
    # gripper change on descend) so the arm is stopped before the jaws close.
    # TODO: implement
    raise NotImplementedError

def release(target, kind: str) -> list:
    """Open the gripper in place at the grasp point."""
    # TODO: implement (mirror of grasp)
    raise NotImplementedError

def lift(target, gripper: float) -> list:
    """[Waypoint] back up to hover over target, slow, keeping `gripper`."""
    # TODO: implement
    raise NotImplementedError

def park(gripper=ARM["gripper_open"]) -> list:
    """[Waypoint] joint-space park pose (arm.park in board.yaml)."""
    # Design: joint-space (q), not Cartesian: park is a known safe pose, no IK needed.
    # TODO: implement
    raise NotImplementedError

## Composite motions

In [ ]:
def pick(target, kind: str) -> list:
    """Hover, descend open, close, lift. Ends at hover holding the piece."""
    # Design: concatenate the primitives; the gripper state must be consistent across the
    # boundaries (open until grasp, closed from grasp on).
    # TODO: implement
    raise NotImplementedError

def place(target, kind: str) -> list:
    """Hover holding the piece, descend, open, lift. Ends at hover, open."""
    # TODO: implement (mirror of pick with release)
    raise NotImplementedError

def move_piece(src, dst, kind: str) -> list:
    """pick(src) + place(dst)."""
    # TODO: implement
    raise NotImplementedError

def capture_to_graveyard(square, kind: str, slot: int) -> list:
    """Remove the piece on `square` to graveyard slot `slot` (placed on the table, z = 0)."""
    # Design: pick on the square, place at graveyard_slot(slot) (a world point, not a square).
    # TODO: implement
    raise NotImplementedError

## Chess move -> motion sequence

Order matters: clear the captured piece first, then move the mover. Castling moves the king
then the rook; en passant removes the pawn *behind* the destination square. Promotion would
swap the pawn for a spare queen from the graveyard (TODO). Uses python-chess (imported here
so the rest of the notebook does not need it).

In [ ]:
def plan_chess_move(board, move, next_grave_slot: int = 0):
    """board: chess.Board BEFORE the move; move: chess.Move.
    Returns (waypoints, slots_used)."""
    # Design:
    # - python-chess does the chess logic: board.piece_at, is_capture, is_en_passant,
    #   is_castling, move.promotion. Import chess inside the function.
    # - Captured piece first (the destination must be empty before the mover arrives):
    #   en passant -> the pawn on (to-file, from-rank), not on to_square; normal capture ->
    #   the piece on to_square. Each capture uses the next graveyard slot.
    # - Then the mover; for castling also the rook (h->f or a->d on the same rank).
    # - Promotion: NotImplementedError for now (pawn out, spare queen in; TODO list).
    # - End with park so the camera sees an unobstructed board.
    # - Raise ValueError if there is no piece on from_square.
    # TODO: implement
    raise NotImplementedError

## Waypoints -> joint targets

IK each Cartesian waypoint (seeded from the previous solution so the arm keeps its
configuration), run the safety check on each target, and fail loudly if anything is wrong.
`densify` then splits every segment into steps no larger than `velocity.max_step_rad`
(linear in joint space) and re-checks each intermediate point, including the step size.

In [ ]:
@dataclass
class JointTarget:
    name: str
    q: np.ndarray
    gripper: float
    slow: bool

class PlanningError(RuntimeError):
    pass

# Tilts (deg from vertical) tried in order until IK converges inside the safety limits.
# Positive leans away from the arm (far ranks), negative toward it (near ranks).
TILT_CANDIDATES_DEG = [0, 10, -10, 20, -20, 30, -30, 35, -35]

def solve_waypoint(wp, seed):
    """IK for one Cartesian waypoint: (q, tilt_deg) or (None, None)."""
    # Design:
    # - Try wp.tilt_deg first, then TILT_CANDIDATES_DEG in order, skipping anything beyond
    #   safety.yaml workspace.max_tilt_deg.
    # - ik_top_down with the safety joint limits (joint_limits()), seeded from `seed` (the
    #   previous waypoint's solution) so the arm keeps the same configuration branch.
    # - First converged solution wins (least tilt first = most top-down grasp).
    # TODO: implement
    raise NotImplementedError

def plan_joint_trajectory(waypoints, q_start=None) -> list:
    """[Waypoint] -> [JointTarget], every target IK-solved and safety-checked.
    Raises PlanningError naming the waypoint on any failure."""
    # Design:
    # - Joint-space waypoints (park) go through as-is (validate joints, skip FK checks).
    # - Cartesian ones: solve_waypoint seeded from the previous IK solution; no solution ->
    #   PlanningError with the point and the tilts tried.
    # - validate_command on each q and gripper_command_ok on the gripper; any problem ->
    #   PlanningError. Fail loudly: never send a partial trajectory to the arm.
    # - Step size between targets is NOT checked here; densify handles that.
    # TODO: implement
    raise NotImplementedError

In [ ]:
def densify(trajectory, max_step=None, q_start=None) -> list:
    """Insert joint-space linear interpolation points so no joint moves more than `max_step`
    rad between consecutive targets; every point is safety-checked (step + pose).
    TODO: Cartesian straight-line interpolation for descend/lift (`slow` segments), and timing
    (time_from_start) from velocity.max_joint_rad_s / max_tool_m_s."""
    # Design:
    # - Default max_step slightly below velocity.max_step_rad (e.g. 95 %) so float rounding
    #   never trips the safety check.
    # - Per segment: n = ceil(largest joint change / max_step) steps, linear in joint space.
    # - Intermediate points keep the PREVIOUS gripper value; only the final point of a segment
    #   gets the new one (the gripper changes once the arm arrives).
    # - validate_command(q, q_prev=...) on every point; name intermediates "<name> (k/n)".
    # Trade-off: joint-space lines are not straight in Cartesian space (fine in transit, not
    # for descend/lift near pieces: see the TODO).
    # TODO: implement
    raise NotImplementedError

### Which squares can we actually reach? (design fact, measured with the reference solution)

`reachability_table()` should print, per square, the tilt that works for both hover and pawn
grasp height (`--` = none).

**Finding:** with the current board.yaml (board center x = 0.215 m, hover height 0.07 m) and
the URDF / safety.yaml wrist_flex limit, **squares c1, d1, e1 and f1 are unreachable at hover
height**: they are so close to the base that the arm would have to fold further than the
wrist limit allows to put the tip 7 cm above them with an approach within 35 deg of vertical.
Every other square is reachable at some tilt. Options (decide on the real arm):
- a lower hover height for rank 1 only (e.g. a per-rank hover height in board.yaml), or
- move the board a little farther from the arm (increase `board.center` x; check that rank 8
  stays reachable and update the Gazebo world, which reads the same file).
(A tilted approach leaning back more than 35 deg would also work geometrically, but it breaks
the "near-vertical near pieces" safety rule.)

In [ ]:
def reachability_table(kind: str = "P") -> str:
    """Text table, rank 8 at the top: per square the max |tilt| (deg) needed for both hover
    and grasp of `kind`, or '--' if either is unreachable."""
    # Design: for every square run solve_waypoint at hover_point and grasp_point (fresh
    # default seed each square so results do not depend on scan order); a square counts as
    # reachable only if both converge. ~64 x 2 IK solves, so do not run on import.
    # TODO: implement, then confirm the c1-f1 finding above.
    raise NotImplementedError

# print(reachability_table())      # ~5 s once implemented

## Execution backends

One interface, several implementations. Only the dry run works here; the rest need hardware
or a running simulator.

In [ ]:
class Backend:
    def execute(self, trajectory):
        """Run a list of JointTarget. Return True on success."""
        raise NotImplementedError

    def gripper_position(self):
        """Measured gripper joint (rad), used by grasp verification."""
        raise NotImplementedError

@dataclass
class DryRunBackend(Backend):
    """Records what would be sent; lets every higher layer be tested without hardware."""
    log: list = field(default_factory=list)
    gripper: float = ARM["gripper_open"]

    def execute(self, trajectory) -> bool:
        # Design: append (name, rounded q, gripper) per target to self.log, track the last
        # gripper command as the "measured" gripper position; always succeed.
        # TODO: implement
        raise NotImplementedError

    def gripper_position(self) -> float:
        # Design: return the last commanded gripper value (an ideal gripper).
        # TODO: implement
        raise NotImplementedError

class ServoBackend(Backend):
    """Real arm through ServoBus (servo_bus.ipynb), guarded by EStop (safety.ipynb)."""
    def execute(self, trajectory):
        # Design: own a ServoBus and an EStop; interpolate each segment at safety.yaml speeds,
        # set_joint_targets() at ~50-100 Hz, wait until within tolerance of each target
        # (read_positions), feed the heartbeat, abort on e-stop.
        # TODO: implement on the real arm
        raise NotImplementedError("ServoBackend")

class GazeboBackend(Backend):
    """Simulated arm through the ros2_control trajectory controllers."""
    def execute(self, trajectory):
        # Design: rclpy action client for /arm_controller/follow_joint_trajectory and
        # /gripper_controller/follow_joint_trajectory (joint names in controllers.yaml), one
        # goal per segment with time_from_start from the speed limits. Pieces are static in
        # Gazebo: attach / teleport the piece on grasp/release (gambit_sim teleport notebook).
        # TODO: implement
        raise NotImplementedError("GazeboBackend")

class LeRobotBackend(Backend):
    """Same targets through LeRobot's SO-101 follower interface."""
    def execute(self, trajectory):
        # Design: send joint targets through LeRobot's follower robot (degrees, its own
        # calibration files), so scripted and learned policies share one execution path.
        # TODO: implement
        raise NotImplementedError("LeRobotBackend")

## Self-check

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - pick("e2", "P") names == ["hover e2", "descend", "grasp", "lift"]; hover z = 0.006 +
#   hover_height; descend z == grasp_point z; grasp closes the gripper
# - move_piece("e2", "e4", "P"): 8 waypoints, ends open, closed while carrying (index 5)
# - capture_to_graveyard("d5", "N", 2) ends over graveyard_slot(2)
# - plan_joint_trajectory(move_piece("e2","e4","P") + park()): 9 finite targets, FK of each
#   within 1 mm of its waypoint
# - densify: more points than the input, last is "park", no joint step > max_step_rad
#   (include the step from the park start pose)
# - DryRunBackend executes and reports the gripper open at the end
# - plan_chess_move on "r3k2r/pppq1ppp/2n5/3pP3/8/8/PPP2PPP/R3K2R w KQkq d6 0 1":
#     e5d6 (en passant) uses 1 slot and starts at "hover d5"; e1g1 (castling) uses 0 slots,
#     has 2 grasps and ends with park
# - a waypoint in c1 at hover height raises PlanningError (the reachability finding)
# Example shape of a test:
# assert [w.name for w in pick("e2", "P")] == ["hover e2", "descend", "grasp", "lift"]
print("Self-check: not written yet (see plan above)")
print("pick_place OK")